In [1]:
from pathlib import Path
import zipfile

def is_within_directory(base: Path, target: Path) -> bool:
    # zip-slip 방지: target이 base 밖으로 나가면 차단
    try:
        target.resolve().relative_to(base.resolve())
        return True
    except Exception:
        return False

def safe_extract_zip(zip_path: Path, out_dir: Path, overwrite=False):
    with zipfile.ZipFile(zip_path, "r") as zf:
        for info in zf.infolist():
            # zip 안의 경로(폴더 포함)
            member = Path(info.filename)

            # 빈 이름/이상한 이름 스킵
            if not info.filename or info.filename.endswith("/"):
                continue

            dest = out_dir / member

            # zip-slip 방지
            if not is_within_directory(out_dir, dest):
                print(f"[SKIP][zip-slip 위험] {zip_path.name} :: {info.filename}")
                continue

            dest.parent.mkdir(parents=True, exist_ok=True)

            # 동일 파일명 충돌 처리
            if dest.exists() and not overwrite:
                stem, suffix = dest.stem, dest.suffix
                k = 1
                while True:
                    new_dest = dest.with_name(f"{stem}__dup{k}{suffix}")
                    if not new_dest.exists():
                        dest = new_dest
                        break
                    k += 1

            # 실제 추출
            with zf.open(info, "r") as src, open(dest, "wb") as dst:
                dst.write(src.read())

def unzip_all_in_date_folders(root_dir, date_names, recurse=True, overwrite=False):
    root = Path(root_dir)

    for dname in date_names:
        date_dir = root / dname
        if not date_dir.exists():
            print(f"[MISS] 폴더 없음: {date_dir}")
            continue

        # 날짜 폴더 내부에서 zip 탐색(하위까지)
        zips = list(date_dir.rglob("*.zip")) if recurse else list(date_dir.glob("*.zip"))
        if not zips:
            print(f"[OK] zip 없음: {date_dir}")
            continue

        print(f"\n=== {dname} : zip {len(zips)}개 발견 ===")
        for zp in sorted(zips):
            print(f" - 풀기: {zp.relative_to(root)}")
            try:
                safe_extract_zip(zp, out_dir=date_dir, overwrite=overwrite)
            except zipfile.BadZipFile:
                print(f"   [FAIL] 손상된 zip: {zp.name}")
            except Exception as e:
                print(f"   [FAIL] {zp.name} / {type(e).__name__}: {e}")

# ====== 사용 예시 ======
# 1) 네 실제 "양상추 날짜/연습용" 폴더 경로로 바꿔줘
ROOT_DIR = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/연습용"   # 예시(윈도우)
DATE_LIST = ["251222", "251223", "251224","251225" ,"251226"]  # 필요하면 "251210"도 추가

unzip_all_in_date_folders(
    root_dir=ROOT_DIR,
    date_names=DATE_LIST,
    recurse=True,       # 날짜 폴더 안 하위폴더까지 zip 찾기
    overwrite=False     # 같은 파일명 있으면 __dup1 붙여서 저장
)


[OK] zip 없음: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/연습용/251222

=== 251223 : zip 3개 발견 ===
 - 풀기: 251223/ai_images_2025-12-27 (1).zip
 - 풀기: 251223/ai_images_2025-12-27 (2).zip
 - 풀기: 251223/ai_images_2025-12-27.zip

=== 251224 : zip 1개 발견 ===
 - 풀기: 251224/ai_images_2025-12-24.zip

=== 251225 : zip 1개 발견 ===
 - 풀기: 251225/ai_images_2025-12-25.zip

=== 251226 : zip 1개 발견 ===
 - 풀기: 251226/ai_images_2025-12-26.zip
